# 14.1 - LLM Call to Tool Calling

Status: VERIFIED

## What Are We Solving?
Tool calling extends a plain LLM API call so the model can request structured function invocations instead of only generating text. Every agent capability depends on this foundation.

## Mental Model
A tool schema is a contract: the LLM sees function names, descriptions, and parameter types, and outputs structured JSON when it decides a tool is needed.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Define Tool Schemas

In [2]:
# Tool definitions (schemas the LLM sees)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. 'Tokyo'"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a math expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression like '2 + 3 * 4'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]

# The actual tool implementations (YOUR code runs these, not the LLM)
def get_weather(city: str) -> str:
    # In production, call a real weather API
    return json.dumps({"city": city, "temp_c": 22, "condition": "clear sky"})

def calculate(expression: str) -> str:
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return json.dumps({"error": "Invalid characters in expression"})
    result = eval(expression)
    return json.dumps({"result": result})

# Tool registry
tool_map = {"get_weather": get_weather, "calculate": calculate}
print("Tools defined:", list(tool_map.keys()))

Tools defined: ['get_weather', 'calculate']


## Tool Calling with Groq

In [3]:
# Send a prompt that should trigger tool calling
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the weather in Tokyo?"}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")

if message.tool_calls:
    for tc in message.tool_calls:
        print(f"\nTool call: {tc.function.name}")
        print(f"Arguments: {tc.function.arguments}")
        
        # Execute the tool application-side
        args = json.loads(tc.function.arguments)
        result = tool_map[tc.function.name](**args)
        print(f"Result: {result}")
else:
    print(f"Text response: {message.content}")

Finish reason: tool_calls

Tool call: get_weather
Arguments: {"city":"Tokyo"}
Result: {"city": "Tokyo", "temp_c": 22, "condition": "clear sky"}


## Complete Tool Calling Cycle

In [4]:
def run_with_tools(user_message: str) -> str:
    """Send a message, execute any tool calls, return final answer."""
    messages = [{"role": "user", "content": user_message}]
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    message = response.choices[0].message
    
    if message.tool_calls:
        # Append assistant message with tool calls
        messages.append(message)
        
        for tc in message.tool_calls:
            args = json.loads(tc.function.arguments)
            result = tool_map[tc.function.name](**args)
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })
        
        # Get final answer after tool execution
        final = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        return final.choices[0].message.content
    
    return message.content

# Test it
answer = run_with_tools("Calculate 42 * 17 + 3")
print(f"Answer: {answer}")

Answer: **42 × 17 + 3 = 717**

Breakdown: 42 × 17 = 714, then 714 + 3 = **717**.


In [5]:
# Verification
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'verified' only"}],
    max_tokens=10
)
assert response.choices[0].message.content is not None
print("VERIFICATION PASSED: Phase 14.1 complete")

VERIFICATION PASSED: Phase 14.1 complete
